<a href="https://colab.research.google.com/github/niclohan/CalculoNumerico/blob/main/gauss_eliminacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eliminação de Gauss + Substituição Regressiva — Animação (Manim)

Este notebook gera uma animação (GIF) que mostra, passo a passo:
1. o sistema linear original;
2. a matriz aumentada sendo triangularizada pela eliminação de Gauss;
3. a substituição regressiva, calculando cada incógnita;
4. o vetor solução final;
5. a interpretação geométrica: cada equação é um plano em 3D, e a solução é o ponto
   onde os três planos se cruzam (com os eixos x, y, z visíveis e a câmera girando).

**Para testar outro sistema:** edite só a célula da Seção 1 (`sistema.py`) — a matriz
`A` e o vetor `b` — e rode o notebook de novo a partir dali. O número de equações, de
etapas da eliminação e de linhas da substituição se ajustam sozinhos.

Usamos a biblioteca **Manim** (Manim Community) porque é feita exatamente pra esse tipo
de animação matemática (equações em LaTeX, transformações entre "cenas", câmera 3D).

## 0. Instalar as dependências (só precisa rodar uma vez por sessão do Colab)

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -qq texlive texlive-latex-extra texlive-fonts-extra texlive-science tipa libcairo2-dev pkg-config ffmpeg
!pip install -q manim

## 1. Sistema linear (altere aqui para testar outro sistema)

`sistema.py` guarda só a matemática (a matriz `A`, o vetor `b`, e as funções de
eliminação de Gauss e substituição regressiva), sem nada de Manim — assim dá pra testar
rapidinho se a conta está certa antes de gastar tempo renderizando a animação.

In [ ]:
%%writefile sistema.py
import numpy as np

# ===== CONFIGURAÇÃO: troque a matriz A e o vetor b para testar outro sistema =====
A = np.array([
    [2, 1, -1],
    [-3, -1, 2],
    [-2, 1, 2],
], dtype=float)
b = np.array([8, -11, -3], dtype=float)
# ===================================================================================


def eliminacao_de_gauss(A, b):
    """Faz a eliminação de Gauss e devolve a lista de matrizes aumentadas [A|b],
    uma para cada etapa (a primeira é o sistema original, a última é triangular superior)."""
    n = len(b)
    M = np.hstack([A, b.reshape(-1, 1)]).astype(float)
    etapas = [M.copy()]
    for col in range(n - 1):
        pivo = M[col, col]
        # ponytail: sem pivoteamento parcial (troca de linhas). Funciona para sistemas
        # cujo pivô natural não é zero, como os pedidos aqui; se precisar de um sistema
        # geral, troque a linha col pela linha com maior |valor| na coluna antes de eliminar.
        if abs(pivo) < 1e-12:
            raise ValueError(f"Pivô nulo na coluna {col}. Este sistema precisa de troca de linhas "
                              "(pivoteamento), que não foi implementado aqui.")
        for lin in range(col + 1, n):
            fator = M[lin, col] / pivo
            M[lin, col:] -= fator * M[col, col:]
        etapas.append(M.copy())
    return etapas


def substituicao_regressiva(M):
    """Recebe a matriz aumentada triangular superior (última etapa da eliminação)
    e devolve a solução x e a lista de passos (i, valor) na ordem calculada."""
    n = M.shape[0]
    x = np.zeros(n)
    passos = []
    for i in reversed(range(n)):
        soma = M[i, i + 1:n] @ x[i + 1:n]
        x[i] = (M[i, -1] - soma) / M[i, i]
        passos.append((i, x[i]))
    return x, passos

### Teste rápido (sem Manim, só pra conferir a conta)

In [ ]:
import importlib, sistema
importlib.reload(sistema)

etapas = sistema.eliminacao_de_gauss(sistema.A, sistema.b)
for i, M in enumerate(etapas):
    print(f"--- etapa {i} ---")
    print(M)

x, passos = sistema.substituicao_regressiva(etapas[-1])
print("\npassos da substituição regressiva:")
for i, v in passos:
    print(f"  x{i} = {v}")
print("\nsolução final:", x)

import numpy as np
assert np.allclose(x, np.linalg.solve(sistema.A, sistema.b))
print("\nok: bate com numpy.linalg.solve")

## 2. Cena da animação

Aqui é só a parte visual (Manim). Ela importa `A`, `b` e as funções de `sistema.py`
— não recalcula nada, só desenha o que já foi calculado.

In [ ]:
%%writefile cena.py
from manim import *
import numpy as np
from sistema import A, b, eliminacao_de_gauss, substituicao_regressiva


def _limpa(v):
    """Troca valores muito perto de zero por 0 exato (evita '-0' feio no texto)."""
    return 0.0 if abs(v) < 1e-9 else v


def eq_tex(coefs, valor):
    """Monta a string LaTeX de uma equação linear, ex.: '2x + y - z = 8'."""
    nomes = ["x", "y", "z", "w"][:len(coefs)]
    termos = []
    for c, nome in zip(coefs, nomes):
        c = _limpa(c)
        if c == 0:
            continue
        sinal = "+" if c >= 0 else "-"
        mag = abs(c)
        coef_str = "" if mag == 1 else f"{mag:g}"
        termos.append(f"{sinal} {coef_str}{nome}")
    corpo = " ".join(termos)
    if corpo.startswith("+ "):
        corpo = corpo[2:]
    elif corpo.startswith("- "):
        corpo = "-" + corpo[2:]
    return f"{corpo} = {valor:g}"


def matriz_tex(M):
    """Matriz aumentada [A|b] em LaTeX, com a barra separando os coeficientes do termo independente."""
    n = M.shape[0]
    fmt = "c" * n + "|c"
    linhas = [" & ".join(f"{_limpa(v):g}" for v in M[i]) for i in range(n)]
    corpo = r" \\ ".join(linhas)
    return r"\left[\begin{array}{%s} %s \end{array}\right]" % (fmt, corpo)


class Gauss(ThreeDScene):
    """Anima a eliminação de Gauss + substituição regressiva para o sistema
    definido em sistema.py (A, b). Para testar outro sistema, edite sistema.py
    — esta cena se ajusta sozinha ao número de equações e de etapas necessárias."""

    def construct(self):
        n = len(b)
        etapas = eliminacao_de_gauss(A, b)
        solucao, passos = substituicao_regressiva(etapas[-1])
        nomes = ["x", "y", "z", "w"][:n]

        # ---------- 1. sistema original ----------
        titulo = Text("Eliminação de Gauss", font_size=40).to_edge(UP)
        equacoes = VGroup(*[
            MathTex(eq_tex(etapas[0][i, :-1], etapas[0][i, -1])) for i in range(n)
        ]).arrange(DOWN, aligned_edge=LEFT, buff=0.4)

        self.play(Write(titulo))
        self.play(Write(equacoes))
        self.wait(1)

        # ---------- 2. eliminação: matriz aumentada mudando a cada etapa ----------
        matriz = MathTex(matriz_tex(etapas[0]))
        self.play(FadeOut(equacoes), FadeIn(matriz))
        self.wait(0.5)

        for k in range(1, len(etapas)):
            nova_matriz = MathTex(matriz_tex(etapas[k]))
            legenda = Text(f"Etapa {k}: zera a coluna {k}", font_size=24).next_to(matriz, DOWN, buff=0.6)
            self.play(Write(legenda))
            self.play(Transform(matriz, nova_matriz))
            self.wait(0.5)
            self.play(FadeOut(legenda))

        caixa = SurroundingRectangle(matriz, color=YELLOW)
        self.play(Create(caixa))
        self.wait(1)
        self.play(FadeOut(caixa), FadeOut(matriz))

        # ---------- 3. substituição regressiva, passo a passo ----------
        subtitulo = Text("Substituição regressiva", font_size=32).to_edge(UP)
        self.play(ReplacementTransform(titulo, subtitulo))

        linhas_sub = VGroup(*[
            MathTex(f"{nomes[i]} = {_limpa(v):g}") for i, v in passos
        ]).arrange(DOWN, buff=0.5)

        for linha in linhas_sub:
            self.play(Write(linha))
            self.wait(0.6)
        self.wait(1)
        self.play(FadeOut(linhas_sub), FadeOut(subtitulo))

        # ---------- 4. solução final ----------
        vetor_tex = r"\vec{x} = \begin{bmatrix} " + r" \\ ".join(f"{_limpa(v):g}" for v in solucao) + r" \end{bmatrix}"
        vetor = MathTex(vetor_tex, font_size=56)
        caixa_final = SurroundingRectangle(vetor, color=GREEN)
        self.play(Write(vetor))
        self.play(Create(caixa_final))
        self.wait(2)

        # ---------- 5. interpretação geométrica (eixos) — só faz sentido com 3 incógnitas ----------
        if n == 3:
            self.play(FadeOut(vetor), FadeOut(caixa_final))
            self.interpretacao_geometrica(solucao)
        else:
            self.wait(1)

    def interpretacao_geometrica(self, solucao):
        eixos = ThreeDAxes(
            x_range=[-6, 6, 2], y_range=[-6, 6, 2], z_range=[-6, 6, 2],
            x_length=8, y_length=8, z_length=8,
        )
        rotulos = eixos.get_axis_labels(x_label="x", y_label="y", z_label="z")

        self.set_camera_orientation(phi=70 * DEGREES, theta=-45 * DEGREES, zoom=0.7)
        self.play(Create(eixos), Write(rotulos))

        # cada equação a0*x + a1*y + a2*z = d é um plano; desenhamos os 3 planos do sistema original
        cores = [BLUE, RED, GREEN]
        for (a0, a1, a2), d, cor in zip(A, b, cores):
            if abs(a2) > 1e-9:
                func = lambda u, v, a0=a0, a1=a1, a2=a2, d=d: np.array([u, v, (d - a0 * u - a1 * v) / a2])
            else:
                func = lambda u, v, a0=a0, a1=a1, d=d: np.array([u, (d - a0 * u) / a1, v])
            plano = Surface(
                func, u_range=[-5, 5], v_range=[-5, 5], resolution=(8, 8),
                fill_color=cor, fill_opacity=0.4, checkerboard_colors=False,
                stroke_width=0.5, stroke_color=cor,
            )
            self.play(FadeIn(plano), run_time=1)

        ponto = Dot3D(point=eixos.c2p(*solucao), color=YELLOW, radius=0.12)
        self.play(FadeIn(ponto))

        rotulo_ponto = MathTex(
            r"\vec{x} = (%.3g,\ %.3g,\ %.3g)" % tuple(solucao), font_size=32, color=YELLOW
        ).to_corner(UR)
        self.add_fixed_in_frame_mobjects(rotulo_ponto)
        self.play(Write(rotulo_ponto))

        self.begin_ambient_camera_rotation(rate=0.2)
        self.wait(6)
        self.stop_ambient_camera_rotation()

## 3. Renderizar o GIF

`-qm` = qualidade média (720p30, rápido o bastante pro Colab). Troque para `-qh` (1080p60)
se quiser mais qualidade e não se importar de esperar mais.

In [ ]:
!manim --format=gif -qm --media_dir media cena.py Gauss

### Localizar, mostrar e baixar o GIF

In [ ]:
import glob, shutil
from IPython.display import Image, display

caminho = glob.glob("media/videos/**/Gauss.gif", recursive=True)[0]
shutil.copy(caminho, "gauss_eliminacao.gif")
display(Image(filename="gauss_eliminacao.gif"))

In [ ]:
try:
    from google.colab import files
    files.download("gauss_eliminacao.gif")
except ImportError:
    print("Não estou no Colab, o gif ficou salvo como gauss_eliminacao.gif")

## Como testar outro sistema

1. Edite a matriz `A` e o vetor `b` na célula da Seção 1 e rode-a de novo (isso
   reescreve `sistema.py`).
2. Rode a célula de teste rápido para conferir a conta.
3. Rode a célula da Seção 2 de novo só se você mudou algo na parte visual (a cena em si
   já se adapta sozinha a qualquer `A`/`b` novos, então normalmente nem precisa reescrever).
4. Rode a Seção 3 de novo para gerar o novo GIF.

A interpretação geométrica (os 3 planos) só aparece para sistemas com exatamente 3
incógnitas — para 2 ou 4+ incógnitas, o resto da animação (equações, eliminação,
substituição, solução) funciona normalmente, só essa última parte é pulada.